In [ ]:
!pip install bacformer

In [ ]:
# Load model directly
from transformers import AutoModelForMaskedLM
model = AutoModelForMaskedLM.from_pretrained("macwiatrak/bacformer-masked-complete-genomes", trust_remote_code=True, dtype="auto")

In [ ]:
import os
os.listdir(".")

In [ ]:
import os, gzip
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
from Bio import SeqIO

import torch
from transformers import AutoModel
from bacformer.pp import protein_seqs_to_bacformer_inputs
from bacformer.modeling import SPECIAL_TOKENS_DICT

In [ ]:
GFF_DIR = "./pangenome"
FAA_PATH = "protein_families.faa"
OUT_DIR = "./PpanggoFormer_embeddings"
ANNOT_PATH = "A_baylyil_families.emapper.annotations"
os.makedirs(OUT_DIR, exist_ok=True)

gff_files = sorted([str(p) for p in Path(GFF_DIR).iterdir()
                    if p.name.endswith((".gff", ".gff.gz", ".gff3", ".gff3.gz"))])

len(gff_files), gff_files[:3]

In [ ]:
def open_maybe_gz(path: str, mode: str = "rt"):
    return gzip.open(path, mode) if path.endswith(".gz") else open(path, mode, encoding="utf-8", errors="replace")

def parse_gff_attrs(attr_str: str) -> dict:
    out = {}
    for item in attr_str.strip().split(";"):
        item = item.strip()
        if not item:
            continue
        if "=" in item:
            k, v = item.split("=", 1)
            out[k.strip()] = v.strip()
    return out

def norm_faa_id(s: str) -> str:
    tok = s.split()[0]
    return tok.split("|")[-1]

def strain_from_gff(gff_path: str) -> str:
    name = Path(gff_path).name
    for suf in [".gz", ".gff3", ".gff"]:
        if name.endswith(suf):
            name = name[: -len(suf)]
    return name

def parse_ppanggolin_gff_cds(gff_path: str) -> pd.DataFrame:
    rows = []
    contig_counts = Counter()

    with open_maybe_gz(gff_path, "rt") as f:
        for line in f:
            if not line or line.startswith("#"):
                continue
            parts = line.rstrip("\n").split("\t")
            if len(parts) < 9 or parts[2] != "CDS":
                continue

            seqid, source, ftype, start, end, score, strand, phase, attrs_str = parts
            a = parse_gff_attrs(attrs_str)

            rows.append(
                {
                    "seqid": seqid,
                    "start": int(start),
                    "end": int(end),
                    "strand": strand,
                    "ID": a.get("ID", ""),
                    "Parent": a.get("Parent", ""),
                    "family": a.get("family", ""),
                    "partition": a.get("partition", ""),
                    "rgp": a.get("rgp", ""),
                    "module": a.get("module", ""),
                    "spot": a.get("spot", ""),
                    "product": a.get("product", ""),
                }
            )
            contig_counts[seqid] += 1

    df = pd.DataFrame(rows)
    if df.empty:
        print("GFF CDS:", 0, "| (vide)")
        return df

    df = df.sort_values(["seqid", "start", "end", "ID"]).reset_index(drop=True)
    print("GFF CDS:", len(df), "| contigs:", df["seqid"].nunique(), "| top contigs:", contig_counts.most_common(5))
    print("GFF missing family:", int((df["family"] == "").sum()))
    return df

def load_faa_queues_by_family(faa_path: str) -> dict[str, list[tuple[str, str]]]:
    fam2q = defaultdict(list)
    total = 0

    with open_maybe_gz(faa_path, "rt") as f:
        for rec in SeqIO.parse(f, "fasta"):
            total += 1
            fam = norm_faa_id(str(rec.id))  # FASTA id = family
            fam2q[fam].append((str(rec.id), str(rec.seq)))

    if total == 0:
        raise ValueError("FAA vide ou non lisible.")

    print("FAA records:", total, "| unique families in FAA:", len(fam2q))
    print("Example FAA ids:", list(fam2q.keys())[:5])
    return fam2q

def match_gff_to_faa_by_family(gff_df: pd.DataFrame, fam2q: dict[str, list[tuple[str, str]]]) -> pd.DataFrame:
    df = gff_df.copy()
    df["faa_id"] = ""
    df["protein_sequence"] = ""
    stats = Counter()

    queues = {k: list(v) for k, v in fam2q.items()}

    for i, r in df.iterrows():
        fam = r["family"]
        if not fam:
            stats["gff_no_family"] += 1
            continue
        if fam not in queues:
            stats["family_not_in_faa"] += 1
            continue
        if not queues[fam]:
            stats["family_exhausted"] += 1
            continue

        faa_id, seq = queues[fam].pop(0)
        df.at[i, "faa_id"] = faa_id
        df.at[i, "protein_sequence"] = seq
        stats["matched"] += 1

    print("Match stats:", dict(stats))
    print("Matched:", int((df["protein_sequence"] != "").sum()), "/", len(df))
    return df

def build_bacformer_genome_info(matched_df: pd.DataFrame):
    df = matched_df[matched_df["protein_sequence"] != ""].copy()
    if df.empty:
        return {"contig_ids": [], "protein_sequence": [], "protein_id": []}, df

    df = df.sort_values(["seqid", "start", "end", "ID"]).reset_index(drop=True)

    contigs = list(df["seqid"].unique())
    contig_to_idx = {c: i for i, c in enumerate(contigs)}
    df["contig_idx"] = df["seqid"].map(contig_to_idx).astype(int)

    df = df.sort_values(["contig_idx", "start", "end", "ID"]).reset_index(drop=True)
    df["protein_idx_flat"] = np.arange(len(df))

    prot_seqs_by_contig = [[] for _ in contigs]
    prot_ids_by_contig  = [[] for _ in contigs]

    for _, r in df.iterrows():
        ci = int(r["contig_idx"])
        prot_seqs_by_contig[ci].append(r["protein_sequence"])
        prot_ids_by_contig[ci].append(r["ID"] if r["ID"] else r["faa_id"])

    genome_info = {
        "contig_ids": contigs,
        "protein_sequence": prot_seqs_by_contig,
        "protein_id": prot_ids_by_contig,
    }
    return genome_info, df

def genes_df_to_payload_dict(genes_df: pd.DataFrame) -> dict:
    cols = ["seqid","start","end","strand","ID","Parent","faa_id","family","partition","rgp","module","spot","product","contig_idx"]
    present = [c for c in cols if c in genes_df.columns]
    d = {c: genes_df[c].tolist() for c in present}
    return d

In [ ]:
faa_cache = load_faa_queues_by_family(FAA_PATH)

In [ ]:
gff_df = parse_ppanggolin_gff_cds(gff_files[20])
matched_df = match_gff_to_faa_by_family(gff_df, faa_cache)
print("match rate:", (matched_df["protein_sequence"] != "").mean())
matched_df["family"].value_counts().head()


In [ ]:
# Charger Bacformer

device = "cuda:0" if torch.cuda.is_available() else "cpu"
dtype = (
    torch.bfloat16 if (device.startswith("cuda") and torch.cuda.is_bf16_supported())
    else (torch.float16 if device.startswith("cuda") else torch.float32)
)

model = AutoModel.from_pretrained(
    "macwiatrak/bacformer-masked-complete-genomes",
    trust_remote_code=True,
).to(device).eval().to(dtype)

BATCH_SIZE = 128
MAX_N_PROTEINS = 6000
PROT_ID = SPECIAL_TOKENS_DICT["PROT_EMB"]

print("device:", device, "| dtype:", dtype, "| PROT_ID:", PROT_ID)

In [ ]:
# Boucle pangenome -> 1 .pt par génome

def run_one_genome(gff_path: str) -> dict:
    strain = strain_from_gff(gff_path)
    out_file = os.path.join(OUT_DIR, f"{strain}_bacformer.pt")
    if os.path.exists(out_file):
        return {"strain": strain, "status": "skip_exists", "out": out_file}

    gff_df = parse_ppanggolin_gff_cds(gff_path)
    if gff_df.empty:
        return {"strain": strain, "status": "skip_empty_gff", "out": out_file}

    matched_df = match_gff_to_faa_by_family(gff_df, faa_cache)
    genome_info, genes_df = build_bacformer_genome_info(matched_df)
    if genes_df.empty:
        return {"strain": strain, "status": "skip_no_matched", "out": out_file}

    inputs = protein_seqs_to_bacformer_inputs(
        genome_info["protein_sequence"],
        device=device,
        batch_size=BATCH_SIZE,
        max_n_proteins=MAX_N_PROTEINS,
    )

    with torch.no_grad():
        outputs = model(**inputs, return_dict=True)

    hidden = outputs["last_hidden_state"][0]      # (L, D)
    stm = inputs["special_tokens_mask"][0]        # (L,)
    prot_mask = (stm == PROT_ID)                  # protéines = positions PROT_EMB

    emb = hidden[prot_mask].detach().cpu().to(torch.float32).numpy()  # (N, D)
    n = emb.shape[0]

    genes_df2 = genes_df.iloc[:n].copy()

    payload = {
        "strain": strain,
        "embeddings": emb.astype(np.float16),
        "special_tokens_mask": stm.detach().cpu(),
        "token_type_ids": inputs["token_type_ids"][0].detach().cpu(),
        "genes": genes_df_to_payload_dict(genes_df2),
    }

    torch.save(payload, out_file)
    return {"strain": strain, "status": "ok", "out": out_file, "shape": payload["embeddings"].shape}


results = []
for gff in gff_files:
    r = run_one_genome(gff)
    results.append(r)
    print(Path(gff).name, "->", r["status"], r.get("shape",""))

pd.DataFrame(results).value_counts("status")

In [ ]:
PT_PATH = "./PpanggoFormer_embeddings/GCA_963520845.1_bacformer.pt"
obj = torch.load(PT_PATH, map_location="cpu", weights_only=False)

type(obj), (obj.keys() if isinstance(obj, dict) else None)

def summarize_pt(d):
    for k, v in d.items():
        if hasattr(v, "shape"):
            print(k, "shape=", tuple(v.shape), "dtype=", getattr(v, "dtype", type(v)))
        elif isinstance(v, dict):
            print(k, "dict keys:", list(v.keys())[:10], "...")
        elif isinstance(v, list):
            print(k, "list len=", len(v))
        else:
            print(k, type(v), str(v)[:120])

summarize_pt(obj)

import pandas as pd
df = pd.DataFrame(obj["genes"])
df.head()


In [ ]:
# Charger tous les .pt + concat (pangenome) pour UMAP

import glob

pt_files = sorted(glob.glob(os.path.join(OUT_DIR, "*_bacformer.pt")))
len(pt_files), pt_files[:3]

In [ ]:
def load_embeddings_and_metadata(pt_path: str) -> tuple[np.ndarray, pd.DataFrame]:
    pt_obj = torch.load(pt_path, map_location="cpu", weights_only=False)

    embeddings = np.asarray(pt_obj["embeddings"])         # (n_proteins, emb_dim)
    metadata_df = pd.DataFrame(pt_obj["genes"])           # (n_proteins, n_fields)
    metadata_df["strain"] = pt_obj["strain"]

    return embeddings, metadata_df


MAX_GENOMES = None
MAX_POINTS = None

all_embeddings = []
all_metadata = []

selected_pt_files = pt_files[: (MAX_GENOMES or len(pt_files))]

for file_idx, pt_file in enumerate(selected_pt_files, start=1):
    emb, meta = load_embeddings_and_metadata(pt_file)
    all_embeddings.append(emb)
    all_metadata.append(meta)

embeddings_all = np.concatenate(all_embeddings, axis=0)
metadata_all = pd.concat(all_metadata, ignore_index=True)

n_proteins, emb_dim = embeddings_all.shape
print(f"Total proteins: {n_proteins} | Embedding dim: {emb_dim} | Metadata shape: {metadata_all.shape}")

if MAX_POINTS is not None and n_proteins > MAX_POINTS:
    sampled_idx = np.random.choice(n_proteins, size=MAX_POINTS, replace=False)
    embeddings_all = embeddings_all[sampled_idx]
    metadata_all = metadata_all.iloc[sampled_idx].reset_index(drop=True)
    print(f"Subsampled proteins: {embeddings_all.shape[0]}")


In [ ]:
# UMAP + Plotly (partition colors fixed)

import umap.umap_ as umap
import plotly.express as px

coords = umap.UMAP(n_neighbors=30, min_dist=0.1, random_state=42).fit_transform(embeddings_all)
metadata_all = metadata_all.copy()
metadata_all["umap1"] = coords[:, 0]
metadata_all["umap2"] = coords[:, 1]

COLOR_BY = "partition"  # "family" | "rgp" | "module" | "spot" | "strain"

metadata_all["partition_norm"] = (
    metadata_all["partition"]
    .fillna("")
    .astype(str)
    .str.strip()
    .str.lower()
    .replace("", "unannotated")
)

metadata_all["color"] = metadata_all["partition_norm"]

color_map = {
    "persistent": "orange",
    "shell": "green",
    "cloud": "blue",
    "unannotated": "lightgray",
    "other": "gray",
}

hover_cols = [c for c in [
    "strain","ID","Parent","family","partition","rgp","module","spot","product",
    "seqid","start","end","strand","contig_idx"
] if c in metadata_all.columns]

fig = px.scatter(
    metadata_all,
    x="umap1",
    y="umap2",
    color="color",
    color_discrete_map=color_map,
    category_orders={"color": ["persistent", "shell", "cloud", "unannotated", "other"]},
    hover_data=hover_cols,
    render_mode="webgl",
    title="UMAP Bacformer — partition (PPanGGOLiN)",
)
fig.update_traces(marker=dict(size=4, opacity=0.85))
fig.update_layout(template="plotly_white")
fig.show()

In [ ]:
counts = metadata_all["partition_norm"].value_counts()

n_persistent = int(counts.get("persistent", 0))
n_shell = int(counts.get("shell", 0))
n_cloud = int(counts.get("cloud", 0))
n_unannotated = int(counts.get("unannotated", 0))

print("Counts:")
print("  persistent:", n_persistent)
print("  shell     :", n_shell)
print("  cloud     :", n_cloud)
print("  unannotated:", n_unannotated)
print("  total     :", len(metadata_all))

In [ ]:
out_html = f"umap_PpanggoFormer_{COLOR_BY}.html"
fig.write_html(out_html, include_plotlyjs="cdn")

from google.colab import files
files.download(out_html)

In [ ]:
# UMAP + Plotly

import umap.umap_ as umap
import plotly.express as px

coords = umap.UMAP(n_neighbors=30, min_dist=0.1).fit_transform(embeddings_all)
metadata_all["umap1"] = coords[:, 0]
metadata_all["umap2"] = coords[:, 1]

COLOR_BY = "rgp"  # "family" | "rgp" | "module" | "spot" | "strain"
TOP_K = 50              # None pour tout afficher

series = metadata_all[COLOR_BY].fillna("").replace("", "Unannotated")
metadata_all["color"] = series if TOP_K is None else series.where(series.isin(series.value_counts().head(TOP_K).index), other="Other")

hover_cols = [c for c in ["strain","ID","Parent","family","partition","rgp","module","spot","product","seqid","start","end","strand","contig_idx"] if c in metadata_all.columns]

fig = px.scatter(
    metadata_all,
    x="umap1",
    y="umap2",
    color="color",
    hover_data=hover_cols,
    render_mode="webgl",
    title=f"UMAP Bacformer — coloré par {COLOR_BY}",
)
fig.update_traces(marker=dict(size=4, opacity=0.85))
fig.update_layout(template="plotly_white")
fig.show()

In [ ]:
out_html = f"umap_PpanggoFormer_{COLOR_BY}.html"
fig.write_html(out_html, include_plotlyjs="cdn")

from google.colab import files
files.download(out_html)

In [ ]:
# UMAP + Plotly

import umap.umap_ as umap
import plotly.express as px

coords = umap.UMAP(n_neighbors=30, min_dist=0.1).fit_transform(embeddings_all)
metadata_all["umap1"] = coords[:, 0]
metadata_all["umap2"] = coords[:, 1]

COLOR_BY = "module"  # "family" | "rgp" | "module" | "spot" | "strain"
TOP_K = 50              # None pour tout afficher

series = metadata_all[COLOR_BY].fillna("").replace("", "Unannotated")
metadata_all["color"] = series if TOP_K is None else series.where(series.isin(series.value_counts().head(TOP_K).index), other="Other")

hover_cols = [c for c in ["strain","ID","Parent","family","partition","rgp","module","spot","product","seqid","start","end","strand","contig_idx"] if c in metadata_all.columns]

fig = px.scatter(
    metadata_all,
    x="umap1",
    y="umap2",
    color="color",
    hover_data=hover_cols,
    render_mode="webgl",
    title=f"UMAP Bacformer — coloré par {COLOR_BY}",
)
fig.update_traces(marker=dict(size=4, opacity=0.85))
fig.update_layout(template="plotly_white")
fig.show()

In [ ]:
out_html = f"umap_PpanggoFormer_{COLOR_BY}.html"
fig.write_html(out_html, include_plotlyjs="cdn")

from google.colab import files
files.download(out_html)

In [ ]:
# UMAP + Plotly

import umap.umap_ as umap
import plotly.express as px

coords = umap.UMAP(n_neighbors=30, min_dist=0.1).fit_transform(embeddings_all)
metadata_all["umap1"] = coords[:, 0]
metadata_all["umap2"] = coords[:, 1]

COLOR_BY = "spot"  # "family" | "rgp" | "module" | "spot" | "strain"
TOP_K = 50              # None pour tout afficher

series = metadata_all[COLOR_BY].fillna("").replace("", "Unannotated")
metadata_all["color"] = series if TOP_K is None else series.where(series.isin(series.value_counts().head(TOP_K).index), other="Other")

hover_cols = [c for c in ["strain","ID","Parent","family","partition","rgp","module","spot","product","seqid","start","end","strand","contig_idx"] if c in metadata_all.columns]

fig = px.scatter(
    metadata_all,
    x="umap1",
    y="umap2",
    color="color",
    hover_data=hover_cols,
    render_mode="webgl",
    title=f"UMAP Bacformer — coloré par {COLOR_BY}",
)
fig.update_traces(marker=dict(size=4, opacity=0.85))
fig.update_layout(template="plotly_white")
fig.show()

In [ ]:
out_html = f"umap_PpanggoFormer_{COLOR_BY}.html"
fig.write_html(out_html, include_plotlyjs="cdn")

from google.colab import files
files.download(out_html)

In [ ]:
# UMAP + Plotly

import umap.umap_ as umap
import plotly.express as px

coords = umap.UMAP(n_neighbors=30, min_dist=0.1).fit_transform(embeddings_all)
metadata_all["umap1"] = coords[:, 0]
metadata_all["umap2"] = coords[:, 1]

COLOR_BY = "strain"  # "family" | "rgp" | "module" | "spot" | "strain"
TOP_K = 50              # None pour tout afficher

series = metadata_all[COLOR_BY].fillna("").replace("", "Unannotated")
metadata_all["color"] = series if TOP_K is None else series.where(series.isin(series.value_counts().head(TOP_K).index), other="Other")

hover_cols = [c for c in ["strain","ID","Parent","family","partition","rgp","module","spot","product","seqid","start","end","strand","contig_idx"] if c in metadata_all.columns]

fig = px.scatter(
    metadata_all,
    x="umap1",
    y="umap2",
    color="color",
    hover_data=hover_cols,
    render_mode="webgl",
    title=f"UMAP Bacformer — coloré par {COLOR_BY}",
)
fig.update_traces(marker=dict(size=4, opacity=0.85))
fig.update_layout(template="plotly_white")
fig.show()

In [ ]:
out_html = f"umap_PpanggoFormer_{COLOR_BY}.html"
fig.write_html(out_html, include_plotlyjs="cdn")

from google.colab import files
files.download(out_html)

In [ ]:
import numpy as np
import umap.umap_ as umap
import plotly.graph_objects as go

# UMAP
coords = umap.UMAP(n_neighbors=30, min_dist=0.1, random_state=42).fit_transform(
    embeddings_all.astype(np.float32)
)

df = metadata_all.copy()
df["umap1"] = coords[:, 0]
df["umap2"] = coords[:, 1]

# couleurs fixes
df["partition_norm"] = (
    df["partition"]
    .fillna("")
    .astype(str)
    .str.strip()
    .str.lower()
    .replace("", "unannotated")
)

color_map = {
    "persistent": "orange",
    "shell": "green",
    "cloud": "blue",
    "unannotated": "lightgray",
}

symbols = [
    "circle", "square", "diamond", "cross", "x",
    "triangle-up", "triangle-down", "triangle-left", "triangle-right",
    "star", "hexagon", "pentagon", "hourglass", "bowtie",
    "circle-open", "square-open", "diamond-open", "cross-open", "x-open",
    "triangle-up-open", "star-open",
]

strains = sorted(df["strain"].astype(str).unique().tolist())
symbol_map = {s: symbols[i % len(symbols)] for i, s in enumerate(strains)}
df["strain_symbol"] = df["strain"].astype(str).map(symbol_map)

# Hover
hover_cols = [c for c in [
    "strain","ID","Parent","faa_id","family","partition","rgp","module","spot","product",
    "seqid","start","end","strand","contig_idx"
] if c in df.columns]

def build_hover_text(sub):
    out = []
    for _, r in sub.iterrows():
        out.append("<br>".join([f"{c}: {r.get(c,'')}" for c in hover_cols]))
    return out

# traces partitions (couleurs), symboles (par point) = strain
fig = go.Figure()

partition_order = ["persistent", "shell", "cloud", "unannotated"]

for part in partition_order:
    sub = df[df["partition_norm"] == part]
    if sub.empty:
        continue

    fig.add_trace(
        go.Scattergl(
            x=sub["umap1"],
            y=sub["umap2"],
            mode="markers",
            name=part,
            legendgroup="partition",
            legendgrouptitle_text="Partition (color)",
            marker=dict(
                color=color_map.get(part, "gray"),
                symbol=sub["strain_symbol"],
                size=4,
                opacity=0.85,
            ),
            text=build_hover_text(sub),
            hoverinfo="text",
            showlegend=True,
        )
    )

for s in strains:
    fig.add_trace(
        go.Scattergl(
            x=[None], y=[None],
            mode="markers",
            name=s,
            legendgroup="strain",
            legendgrouptitle_text="Genome (symbol)",
            marker=dict(
                symbol=symbol_map[s],
                size=10,
                color="black",
                opacity=0.9,
            ),
            hoverinfo="skip",
            showlegend=True,
        )
    )

fig.update_layout(
    template="plotly_white",
    width=1200,
    height=800,
    title="UMAP Bacformer — couleurs=partition, symboles=génomes",
    legend=dict(
        x=1.02, y=1.0, xanchor="left", yanchor="top",  # <-- à droite
        borderwidth=0,
        itemsizing="constant",
    ),
    margin=dict(r=320),
)

fig.show()

In [ ]:
import io
import pandas as pd

def read_emapper_annotations(path: str) -> pd.DataFrame:
    header = None
    data_lines = []
    with open(path, "r", encoding="utf-8", errors="replace") as f:
        for line in f:
            if line.startswith("#"):
                if line.lower().startswith("#query"):
                    header = line.lstrip("#").rstrip("\n").split("\t")
                continue
            if line.strip():
                data_lines.append(line)

    if header is None:
        raise ValueError("Header '#query ...' introuvable dans emapper.annotations.")

    df = pd.read_csv(
        io.StringIO("".join(data_lines)),
        sep="\t",
        names=header,
        dtype=str,
        keep_default_na=False,
    )

    if "query" not in df.columns:
        for c in df.columns:
            if c.lower().strip("#") == "query":
                df = df.rename(columns={c: "query"})
                break
    if "query" not in df.columns:
        raise ValueError("Colonne 'query' introuvable.")

    df["query"] = df["query"].astype(str)
    return df


def norm_id(x: str) -> str:
    x = str(x).split()[0]
    return x.split("|")[-1]


def merge_emapper_on_metadata(metadata_all: pd.DataFrame, annot_path: str) -> pd.DataFrame:
    ann = read_emapper_annotations(annot_path)
    ann["query_norm"] = ann["query"].map(norm_id)

    df = metadata_all.copy()

    candidate_keys = [c for c in ["faa_id", "ID", "Parent", "protein_id"] if c in df.columns]
    if not candidate_keys:
        raise ValueError("Aucune colonne ID trouvée dans metadata_all (attendu: faa_id/ID/Parent/protein_id).")

    best = None
    for k in candidate_keys:
        keys = df[k].map(norm_id)
        hit_rate = keys.isin(set(ann["query_norm"])).mean()
        if best is None or hit_rate > best[1]:
            best = (k, hit_rate)

    merge_key, hit_rate = best
    print("Best merge key:", merge_key, "| hit_rate:", round(hit_rate, 4))

    df["_k"] = df[merge_key].map(norm_id)
    df_merged = (
        df.merge(
            ann.drop(columns=["query"]),
            left_on="_k",
            right_on="query_norm",
            how="left",
        )
        .drop(columns=["_k", "query_norm"])
    )

    print("Merged rows:", len(df_merged))
    return df_merged

In [ ]:
metadata_eggnog = merge_emapper_on_metadata(metadata_all, ANNOT_PATH)

metadata_eggnog.head()

In [ ]:
coords = umap.UMAP(n_neighbors=30, min_dist=0.1, random_state=42).fit_transform(embeddings_all)
metadata_eggnog["umap1"] = coords[:, 0]
metadata_eggnog["umap2"] = coords[:, 1]

COLOR_BY = "COG_category"
TOP_K = 50

series = metadata_eggnog[COLOR_BY].fillna("").replace("", "Unannotated")
metadata_eggnog["color"] = series if TOP_K is None else series.where(
    series.isin(series.value_counts().head(TOP_K).index),
    other="Other",
)

hover_cols = [c for c in [
    "strain","ID","Parent","faa_id","family","partition","seqid","start","end",
    "COG_category","eggNOG_OGs","Description","KEGG_ko","GO"
] if c in metadata_eggnog.columns]

fig = px.scatter(
    metadata_eggnog,
    x="umap1",
    y="umap2",
    color="color",
    hover_data=hover_cols,
    render_mode="webgl",
    title=f"UMAP Bacformer — EggNOG coloré par {COLOR_BY}",
)
fig.update_traces(marker=dict(size=4, opacity=0.85))
fig.update_layout(template="plotly_white")
fig.show()

In [ ]:
coords = umap.UMAP(n_neighbors=30, min_dist=0.1, random_state=42).fit_transform(embeddings_all)
metadata_eggnog["umap1"] = coords[:, 0]
metadata_eggnog["umap2"] = coords[:, 1]

COLOR_BY = "PFAMs"
TOP_K = 50

series = metadata_eggnog[COLOR_BY].fillna("").replace("", "Unannotated")
metadata_eggnog["color"] = series if TOP_K is None else series.where(
    series.isin(series.value_counts().head(TOP_K).index),
    other="Other",
)

hover_cols = [c for c in [
    "strain","ID","Parent","faa_id","family","partition","seqid","start","end",
    "COG_category","eggNOG_OGs","Description","KEGG_ko","GO"
] if c in metadata_eggnog.columns]

fig = px.scatter(
    metadata_eggnog,
    x="umap1",
    y="umap2",
    color="color",
    hover_data=hover_cols,
    render_mode="webgl",
    title=f"UMAP Bacformer — EggNOG coloré par {COLOR_BY}",
)
fig.update_traces(marker=dict(size=4, opacity=0.85))
fig.update_layout(template="plotly_white")
fig.show()

In [ ]:
import umap.umap_ as umap
import plotly.express as px

coords = umap.UMAP(n_neighbors=30, min_dist=0.1, random_state=42).fit_transform(embeddings_all)
metadata_eggnog["umap1"] = coords[:, 0]
metadata_eggnog["umap2"] = coords[:, 1]

def first_item(x: str, sep: str = ",") -> str:
    x = "" if x is None else str(x)
    x = x.strip()
    if not x or x in ("-", "NA"):
        return "Unannotated"
    return x.split(sep)[0].strip()

def plot_umap(
    field: str,
    sep: str = ",",
    top_k: int | None = 50,
    title: str | None = None,
    export_html: bool = False,
    prefix: str = "umap_ppanggolin",
):
    df = metadata_eggnog

    if field not in df.columns:
        raise ValueError(f"Colonne '{field}' absente. Colonnes dispo: {df.columns.tolist()}")

    label = df[field].map(lambda x: first_item(x, sep=sep))

    if top_k is None:
        color = label
    else:
        keep = set(label.value_counts().head(top_k).index)
        color = label.where(label.isin(keep), other="Other")

    hover_cols = [c for c in ["strain","ID","faa_id","Parent","family","partition","product",field,"seqid","start","end","strand","contig_idx"] if c in df.columns]

    fig = px.scatter(
        df.assign(color=color),
        x="umap1",
        y="umap2",
        color="color",
        hover_data=hover_cols,
        render_mode="webgl",
        title=title or f"UMAP Bacformer — {field}",
    )
    fig.update_traces(marker=dict(size=5, opacity=0.85))
    fig.update_layout(template="plotly_white")
    fig.show()

    if export_html:
        import re
        safe_field = re.sub(r"[^a-zA-Z0-9_-]+", "_", field).strip("_")
        safe_topk = "all" if top_k is None else f"top{int(top_k)}"
        out_html = f"{prefix}_{safe_field}_{safe_topk}.html"
        fig.write_html(out_html, include_plotlyjs="cdn", full_html=True)
        print("Saved:", out_html)

    return fig

fig_cog = plot_umap("COG_category", sep=",", top_k=50, export_html=True)

if "PFAMs" in metadata_eggnog.columns:
    fig_pfam = plot_umap("PFAMs", sep=",", top_k=50, export_html=True)

if "GOs" in metadata_eggnog.columns:
    fig_go = plot_umap("GOs", sep=",", top_k=50, export_html=True)

for k in ["KEGG_ko", "KEGG_Pathway", "KEGG_Module"]:
    if k in metadata_eggnog.columns:
        fig_kegg = plot_umap(k, sep=",", top_k=50, export_html=True)
        break

In [ ]:
# UMAP interactive chooser + Export HTML

import re
import numpy as np
import pandas as pd
import umap.umap_ as umap
import plotly.express as px

import ipywidgets as widgets
from IPython.display import display
from google.colab import files
from google.colab import output
output.enable_custom_widget_manager()

import plotly.io as pio
pio.renderers.default = "colab"



metadata_df = metadata_eggnog

# Calcul UMAP une seule fois
coords = umap.UMAP(n_neighbors=30, min_dist=0.1, random_state=42).fit_transform(embeddings_all)
metadata_df = metadata_df.copy()
metadata_df["umap1"] = coords[:, 0]
metadata_df["umap2"] = coords[:, 1]

# CONFIG
TOP_K_DEFAULT = 50     # 0 = show all
SEP_DEFAULT = ","      # PFAMs/GOs/KEGG often ',' or ';'

def first_item(x: str, sep: str = ",") -> str:
    x = "" if x is None else str(x)
    x = x.strip()
    if not x or x in ("-", "NA"):
        return "Unannotated"
    return x.split(sep)[0].strip()

def build_color_series(df: pd.DataFrame, field: str, sep: str, top_k: int) -> pd.Series:
    s = df[field] if field in df.columns else pd.Series(["Unannotated"] * len(df))
    label = s.map(lambda v: first_item(v, sep=sep))

    if top_k == 0:
        return label
    keep = set(label.value_counts().head(top_k).index)
    return label.where(label.isin(keep), other="Other")

def make_umap_figure(df: pd.DataFrame, field: str, sep: str, top_k: int):
    color = build_color_series(df, field, sep=sep, top_k=top_k)

    hover_cols = [c for c in [
        "strain","ID","faa_id","Parent","family","partition","rgp","module","spot","product",
        field,"seqid","start","end","strand","contig_idx"
    ] if c in df.columns]

    fig = px.scatter(
        df.assign(color=color),
        x="umap1",
        y="umap2",
        color="color",
        hover_data=hover_cols,
        render_mode="webgl",
        title=f"UMAP Bacformer — color_by={field}",
    )
    fig.update_traces(marker=dict(size=5, opacity=0.85))
    fig.update_layout(template="plotly_white")
    return fig

def safe_filename(s: str) -> str:
    s = re.sub(r"[^A-Za-z0-9_.-]+", "_", s)
    return s.strip("_")[:120] or "plot"

reserved = {"umap1", "umap2", "protein_sequence", "embeddings"}
fields = [c for c in metadata_df.columns if c not in reserved]

preferred = ["COG_category", "PFAMs", "GOs", "KEGG_ko", "KEGG_Pathway", "KEGG_Module",
             "partition", "family", "rgp", "module", "spot", "strain"]
fields_sorted = [c for c in preferred if c in fields] + [c for c in fields if c not in preferred]
if not fields_sorted:
    raise ValueError("Aucun champ disponible pour color_by.")

# Widgets
dd = widgets.Dropdown(options=fields_sorted, value=fields_sorted[0], description="color_by:",
                      layout=widgets.Layout(width="420px"))
sep_w = widgets.Text(value=SEP_DEFAULT, description="sep:",
                     layout=widgets.Layout(width="180px"))
topk_w = widgets.IntText(value=TOP_K_DEFAULT, description="top_k (0=all):",
                         layout=widgets.Layout(width="230px"))

btn_show = widgets.Button(description="Afficher", button_style="primary")
btn_export = widgets.Button(description="Exporter HTML", button_style="success")

out = widgets.Output()
state = {"fig": None}

def refresh(_=None):
    with out:
        out.clear_output(wait=True)
        fig = make_umap_figure(metadata_df, dd.value, sep=sep_w.value, top_k=int(topk_w.value))
        state["fig"] = fig
        fig.show()

def export_html(_=None):
    fig = state.get("fig")
    if fig is None:
        refresh()
        fig = state.get("fig")

    fname = f"umap_{safe_filename(dd.value)}_top{int(topk_w.value)}.html"
    fig.write_html(fname, include_plotlyjs="cdn", full_html=True)
    files.download(fname)

btn_show.on_click(refresh)
btn_export.on_click(export_html)

display(widgets.HBox([dd, topk_w, sep_w, btn_show, btn_export]))
display(out)

refresh()
